In [1]:
import pandas as pd
import re
import docx
import pdfplumber
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML


def extract_resume_text(file_path):
    text = ""
    try:
        if file_path.lower().endswith(".pdf"):
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    text += page.extract_text() or ""
        elif file_path.lower().endswith(".docx"):
            doc = docx.Document(file_path)
            for para in doc.paragraphs:
                text += para.text
        else:
            raise ValueError("Unsupported file format")
    except Exception as e:
        print("Error reading resume:", e)
        return ""

    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text


def detect_location(resume_text):
    locations = [
        "india", "mumbai", "delhi", "bangalore", "pune",
        "hyderabad", "chennai", "kolkata", "ahmedabad"
    ]
    for loc in locations:
        if loc in resume_text:
            return "India"
    return "India"


def clean_location(row):
    company = str(row["company_name"]).lower()
    location = str(row["location"]).lower()

    location = location.replace(company, "")
    location = location.replace("remote in", "remote")
    location = re.sub(r"\s+", " ", location)

    return location.strip().title()


jobs_df = pd.read_csv("jobs.csv")

jobs_df["location"] = jobs_df.apply(clean_location, axis=1)

jobs_df["text_data"] = (
    jobs_df["job_title"].fillna("") + " " +
    jobs_df["job_description"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")


def recommend_jobs_from_path(file_path, top_n=10):
    resume_text = extract_resume_text(file_path)
    if not resume_text.strip():
        return pd.DataFrame()

    # Priority 1: India jobs
    india_jobs = jobs_df[
        jobs_df["location"].str.contains("india", case=False, na=False)
    ]

    # Priority 2: Remote jobs
    remote_jobs = jobs_df[
        jobs_df["location"].str.contains("remote", case=False, na=False)
    ]

    # Decide which to use
    if not india_jobs.empty:
        candidate_jobs = india_jobs
    elif not remote_jobs.empty:
        candidate_jobs = remote_jobs
    else:
        candidate_jobs = jobs_df.copy()  # fallback: global jobs

    all_text = candidate_jobs["text_data"].tolist() + [resume_text]
    tfidf_matrix = vectorizer.fit_transform(all_text)

    similarity = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])
    top_indices = similarity[0].argsort()[::-1][:top_n]

    result = candidate_jobs.iloc[top_indices][
        ["job_title", "company_name", "location", "job_url"]
    ]

    result["job_url"] = result["job_url"].apply(
        lambda x: f'<a href="{x}" target="_blank">Apply Here</a>'
    )

    return result

resume_path = input("Enter resume file path (PDF/DOCX): ")
output = recommend_jobs_from_path(resume_path, top_n=10)

if not output.empty:
    display(HTML(output.to_html(escape=False, index=False)))
else:
    print("No job recommendations available.")


Enter resume file path (PDF/DOCX):  "C:\Users\Sunil\Desktop\agent\job recommdation\uploads\Nandini_Pathak_Resume (2).pdf"


Error reading resume: Unsupported file format
No job recommendations available.
